In [ ]:
# the scripts in here are for printing images of each dataset fro each modality for a good axial slice with significant proportion f 
# of the tumour present. 
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
import glob

# Define datasets
datasets = ['ISLES','WMH','BRATS', 'ISLES2022','TBI','ATLAS','MSSEG','TUMOUR2']
data_folder = '/home/magd6292/Documents/wentian_clone/MultiUnet/data'
output_folder = '/home/magd6292/Documents/wentian_clone/MultiUnet/dataset_samples'

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

In [37]:
def find_matching_files(dataset_name, data_folder):
    """Find matching image and label files for a dataset"""
    dataset_path = os.path.join(data_folder, dataset_name)
    images_path = os.path.join(dataset_path, 'Images')
    labels_path = os.path.join(dataset_path, 'Labels')
    
    if not os.path.exists(images_path) or not os.path.exists(labels_path):
        print(f"Warning: Images or Labels folder not found for {dataset_name}")
        return []
    
    # Get all image and label files
    image_files = sorted(glob.glob(os.path.join(images_path, '*.nii.gz')))
    label_files = sorted(glob.glob(os.path.join(labels_path, '*.nii.gz')))
    
    matched_pairs = []
    
    # Special handling for ISLES and ISLES2022 - match by order in sorted list
    if dataset_name in ['ISLES', 'ISLES2022']:
        print(f"  🔄 Using ORDER-BASED matching for {dataset_name}")
        
        # For ISLES: match _image.nii.gz with _label.nii.gz by order
        if dataset_name == 'ISLES':
            image_files = [f for f in image_files if '_image.nii.gz' in f]
            label_files = [f for f in label_files if '_label.nii.gz' in f]
        
        # For ISLES2022: match patient_XXXX.nii.gz with label_patient_XXXX.nii.gz by order
        elif dataset_name == 'ISLES2022':
            # Images are patient_XXXX.nii.gz, labels are label_patient_XXXX.nii.gz
            image_files = [f for f in image_files if not os.path.basename(f).startswith('label_')]
            label_files = [f for f in label_files if os.path.basename(f).startswith('label_')]
        
        # Match by order (both lists should be same length and correspond)
        min_length = min(len(image_files), len(label_files))
        for i in range(min_length):
            matched_pairs.append((image_files[i], label_files[i]))
            
        print(f"  ✅ Matched {len(matched_pairs)} pairs by order")
        if len(matched_pairs) > 0:
            print(f"  📁 Example: {os.path.basename(matched_pairs[0][0])} -> {os.path.basename(matched_pairs[0][1])}")
        
    else:
        # Original pattern-based matching for other datasets
        for image_file in image_files:
            image_basename = os.path.basename(image_file)
            
            # Try different naming patterns for labels
            label_patterns = [
                # Remove common suffixes and add label suffixes
                image_basename.replace('_normed_on_mask.nii.gz', 'merged.nii.gz'),  # BRATS
                image_basename.replace('_normed.nii.gz', '_label_trimmed.nii.gz'),  # ATLAS
                image_basename.replace('.nii.gz', '_label.nii.gz'),  # Generic
                image_basename.replace('.nii.gz', '_seg.nii.gz'),   # Alternative
                image_basename  # Same name
            ]
            
            for pattern in label_patterns:
                label_file = os.path.join(labels_path, pattern)
                if os.path.exists(label_file):
                    matched_pairs.append((image_file, label_file))
                    break
    
    return matched_pairs

# Test the function
for dataset in datasets:
    pairs = find_matching_files(dataset, data_folder)
    print(f"{dataset}: Found {len(pairs)} matching pairs")
    if len(pairs) > 0:
        print(f"  Example: {os.path.basename(pairs[0][0])} -> {os.path.basename(pairs[0][1])}")
    print()


  🔄 Using ORDER-BASED matching for ISLES
  ✅ Matched 28 pairs by order
  📁 Example: 02_image.nii.gz -> 02_label.nii.gz
ISLES: Found 28 matching pairs
  Example: 02_image.nii.gz -> 02_label.nii.gz

WMH: Found 60 matching pairs
  Example: train_0_u.nii.gz -> train_0_u.nii.gz

BRATS: Found 484 matching pairs
  Example: BRATS_001_normed_on_mask.nii.gz -> BRATS_001merged.nii.gz

  🔄 Using ORDER-BASED matching for ISLES2022
  ✅ Matched 0 pairs by order
ISLES2022: Found 0 matching pairs

TBI: Found 281 matching pairs
  Example: CENTER-TBI-2020-6_Sub-001-6ban548_Site-06-a72b20.nii.gz -> CENTER-TBI-2020-6_Sub-001-6ban548_Site-06-a72b20.nii.gz

ATLAS: Found 654 matching pairs
  Example: sub-r001s001_normed.nii.gz -> sub-r001s001_label_trimmed.nii.gz

MSSEG: Found 53 matching pairs
  Example: test_c01_p01.nii.gz -> test_c01_p01.nii.gz

TUMOUR2: Found 57 matching pairs
  Example: TUM_302_highres.nii.gz -> TUM_302_highres.nii.gz



In [38]:
def find_best_axial_tumor_slice_with_config(image_path, label_path):
    """Find the best axial slice with proper modality names from config and patient info"""
    try:
        # Load the image and label
        image_nii = nib.load(image_path)
        label_nii = nib.load(label_path)
        
        image_data = image_nii.get_fdata()
        label_data = label_nii.get_fdata()
        
        # Handle different dimensions for labels
        if len(label_data.shape) == 4:
            label_data = label_data[:, :, :, 0]  # Take first channel
        
        # Focus on axial slices (z-axis, typically axis 2)
        # Find slices with good tumor visibility in the middle region of the brain
        z_slices = label_data.shape[2]
        
        # Look in the middle 60% of slices (avoid top and bottom)
        start_slice = int(z_slices * 0.2)
        end_slice = int(z_slices * 0.8)
        
        best_slice_idx = start_slice
        max_score = 0
        
        for i in range(start_slice, end_slice):
            slice_data = label_data[:, :, i]
            tumor_area = np.sum(slice_data > 0)
            
            # Score based on tumor area and position (prefer middle slices)
            middle_bonus = 1.0 - abs(i - z_slices/2) / (z_slices/2)  # Higher score for middle slices
            score = tumor_area * (1 + 0.5 * middle_bonus)  # Bonus for being in middle
            
            if score > max_score and tumor_area > 100:  # Minimum tumor size threshold
                max_score = score
                best_slice_idx = i
        
        # Extract the best axial slice for all modalities
        label_slice = label_data[:, :, best_slice_idx]
        tumor_area = np.sum(label_slice > 0)
        
        # Get dataset name from path for proper modality naming
        dataset_name = None
        for ds in ['BRATS', 'ATLAS', 'MSSEG', 'ISLES', 'TBI', 'ISLES2022', 'TUMOUR2','WMH']:
            if ds in image_path:
                dataset_name = ds
                break
        
        # Get patient/file name for verification
        patient_name = os.path.basename(image_path).replace('.nii.gz', '')
        
        # Handle multi-modal images (4D) or single modal (3D)
        if len(image_data.shape) == 4:
            # Multi-modal: return all modalities
            image_slices = []
            num_modalities = image_data.shape[3]
            
            # Define modalities based on config.py - exact order as they are stacked
            dataset_modalities = {
                'BRATS': ["FLAIR", "T1", "T1c", "T2"],
                'ATLAS': ["T1"],
                'MSSEG': ['FLAIR', "T1", "T1c", "T2", "PD"],
                'ISLES': ["FLAIR", "T1", "T2", "DWI"],
                'TBI': ["FLAIR", "T1", "T2", "SWI"],
                'ISLES2022': ['ADC', 'DWI', 'FLAIR'],  # Corrected: ADC, DWI, FLAIR (not T1, T2)
                'TUMOUR2': ['T1'],
                'WMH': ['FLAIR','T1']
            }
            
            if dataset_name and dataset_name in dataset_modalities:
                config_modalities = dataset_modalities[dataset_name]
                # Use the config modalities, but only up to the number we actually have
                modality_names = config_modalities[:num_modalities]
                # If we have more modalities than expected, fill with generic names
                if num_modalities > len(config_modalities):
                    for i in range(len(config_modalities), num_modalities):
                        modality_names.append(f'Modality_{i+1}')
            else:
                # Fallback to generic names
                modality_names = [f'Modality_{i+1}' for i in range(num_modalities)]
            
            for mod_idx in range(num_modalities):
                image_slice = image_data[:, :, best_slice_idx, mod_idx]
                image_slices.append(image_slice)
                
            return image_slices, modality_names, label_slice, best_slice_idx, tumor_area, patient_name
        else:
            # Single modal: return as list for consistency
            image_slice = image_data[:, :, best_slice_idx]
            return [image_slice], ['Single'], label_slice, best_slice_idx, tumor_area, patient_name
        
    except Exception as e:
        print(f"Error processing {image_path}: {str(e)}")
        return None, None, None, 0, 0, None


In [39]:
def save_uniform_size_images_no_crop(image_slice, label_slice, dataset_name, modality_name, slice_idx, output_folder):
    """Save images with EXACTLY the same dimensions - resize but NEVER crop the brain"""
    
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    if image_slice is not None:
        # Rotate anticlockwise by 90 degrees
        image_slice_rotated = np.rot90(image_slice, k=1)  # k=1 for 90° anticlockwise
        label_slice_rotated = np.rot90(label_slice, k=1)
        
        # Normalize image slice for better visualization
        image_normalized = (image_slice_rotated - np.min(image_slice_rotated)) / (np.max(image_slice_rotated) - np.min(image_slice_rotated) + 1e-8)
        
        # Create filename
        filename_base = f"{dataset_name}_{modality_name}" if modality_name else dataset_name
        
        # TARGET DIMENSIONS - All images will be EXACTLY this size
        TARGET_WIDTH = 512
        TARGET_HEIGHT = 512
        
        # Resize using scipy.ndimage to preserve full brain content
        from scipy.ndimage import zoom
        
        # Calculate zoom factors
        zoom_factor_y = TARGET_HEIGHT / image_normalized.shape[0]
        zoom_factor_x = TARGET_WIDTH / image_normalized.shape[1]
        
        # Use the SAME zoom factor for both dimensions to maintain aspect ratio
        # Use the smaller zoom factor to ensure the entire brain fits
        zoom_factor = min(zoom_factor_y, zoom_factor_x)
        
        # Resize with uniform scaling (no distortion)
        image_resized = zoom(image_normalized, [zoom_factor, zoom_factor], order=1)
        label_resized = zoom(label_slice_rotated, [zoom_factor, zoom_factor], order=0)  # Nearest neighbor for labels
        
        # Create target-sized arrays and center the resized image
        final_image = np.zeros((TARGET_HEIGHT, TARGET_WIDTH))
        final_label = np.zeros((TARGET_HEIGHT, TARGET_WIDTH))
        
        # Calculate centering offsets
        y_offset = (TARGET_HEIGHT - image_resized.shape[0]) // 2
        x_offset = (TARGET_WIDTH - image_resized.shape[1]) // 2
        
        # Place the resized image in the center
        final_image[y_offset:y_offset+image_resized.shape[0], 
                   x_offset:x_offset+image_resized.shape[1]] = image_resized
        final_label[y_offset:y_offset+label_resized.shape[0], 
                   x_offset:x_offset+label_resized.shape[1]] = label_resized
        
        # Save image - EXACT DIMENSIONS using direct array creation
        fig, ax = plt.subplots(figsize=(5.12, 5.12))  # 5.12 inches at 100 DPI = 512 pixels
        ax.imshow(final_image, cmap='gray', aspect='equal')
        ax.set_xlim(0, TARGET_WIDTH)
        ax.set_ylim(TARGET_HEIGHT, 0)
        ax.axis('off')
        
        # Remove ALL margins and padding
        plt.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0, hspace=0)
        
        image_path = os.path.join(output_folder, f'{filename_base}_image.png')
        plt.savefig(image_path, dpi=100, facecolor='black', edgecolor='none')
        plt.close()
        
        # Save label - EXACT DIMENSIONS with black background
        fig, ax = plt.subplots(figsize=(5.12, 5.12))  # 5.12 inches at 100 DPI = 512 pixels
        
        # Black background
        ax.imshow(np.zeros((TARGET_HEIGHT, TARGET_WIDTH)), cmap='gray', aspect='equal')
        
        # Overlay tumor in red
        if np.sum(final_label) > 0:
            tumor_mask = final_label > 0
            masked_label = np.ma.masked_where(~tumor_mask, final_label)
            ax.imshow(masked_label, cmap='Reds', alpha=1.0, aspect='equal', vmin=0, vmax=1)
        
        ax.set_xlim(0, TARGET_WIDTH)
        ax.set_ylim(TARGET_HEIGHT, 0)
        ax.axis('off')
        
        # Remove ALL margins and padding
        plt.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0, hspace=0)
        
        label_path = os.path.join(output_folder, f'{filename_base}_label.png')
        plt.savefig(label_path, dpi=100, facecolor='black', edgecolor='none')
        plt.close()
        
        return image_path, label_path
    
    return None, None


In [40]:
# Main processing loop for individual image and label files (NO POSTER)
results = {}

print("Creating individual image and label files for each dataset...")
print("=" * 50)

for dataset in datasets:
    print(f"\nProcessing {dataset}...")
    
    # Find matching files
    pairs = find_matching_files(dataset, data_folder)
    
    if len(pairs) == 0:
        print(f"  No matching pairs found for {dataset}")
        continue
    
    # Try multiple files to find the best one with visible tumors
    best_image = None
    best_result = None
    max_tumor_area = 0
    
    for i, (image_path, label_path) in enumerate(pairs[:5]):  # Check first 5 files
        print(f"  Checking file {i+1}: {os.path.basename(image_path)}")
        
        # Find the best axial slice
        result = find_best_axial_tumor_slice_with_config(image_path, label_path)
        if len(result) == 6 and result[0] is not None:
            image_slices, modality_names, label_slice, slice_idx, tumor_area, patient_name = result
            # Use first modality for this old cell
            image_slice = image_slices[0] if image_slices else None
        else:
            image_slice, label_slice, slice_idx, tumor_area = None, None, 0, 0
        
        if image_slice is not None and tumor_area > max_tumor_area:
            max_tumor_area = tumor_area
            best_image = (image_slice, label_slice, slice_idx, tumor_area)
            best_result = (image_path, label_path)
            print(f"    Found better slice {slice_idx} with tumor area {tumor_area}")
    
    if best_image is not None:
        image_slice, label_slice, slice_idx, tumor_area = best_image
        image_path, label_path = best_result
        
        # Save individual image and label files (NO POSTER)
        img_path, lbl_path = save_uniform_size_images_no_crop(
            image_slice, label_slice, dataset, 'Single', slice_idx, output_folder
        )
        
        results[dataset] = {
            'original_image': image_path,
            'original_label': label_path,
            'slice_index': slice_idx,
            'tumor_area': tumor_area,
            'image_png': img_path,
            'label_png': lbl_path
        }
        
        print(f"  ✓ Created images for {dataset}")
        print(f"    Axial slice {slice_idx} with tumor area {tumor_area}")
        print(f"    Image saved: {img_path}")
        print(f"    Label saved: {lbl_path}")
    else:
        print(f"  ✗ No suitable tumor found for {dataset}")

print(f"\n" + "=" * 50)
print(f"IMAGE CREATION COMPLETE")
print(f"=" * 50)
print(f"Successfully created images for {len(results)} datasets:")
for dataset, info in results.items():
    print(f"  {dataset}: axial slice {info['slice_index']} - tumor area: {info['tumor_area']}")

print(f"\nImage files saved to: {output_folder}")
print("Files created:")
for dataset in results.keys():
    print(f"  - {dataset}_Single_image.png (brain image)")
    print(f"  - {dataset}_Single_label.png (tumor mask)")


Creating individual image and label files for each dataset...

Processing ISLES...
  🔄 Using ORDER-BASED matching for ISLES
  ✅ Matched 28 pairs by order
  📁 Example: 02_image.nii.gz -> 02_label.nii.gz
  Checking file 1: 02_image.nii.gz
    Found better slice 78 with tumor area 862
  Checking file 2: 03_image.nii.gz


  Checking file 3: 04_image.nii.gz
    Found better slice 90 with tumor area 4483
  Checking file 4: 06_image.nii.gz
  Checking file 5: 07_image.nii.gz
  ✓ Created images for ISLES
    Axial slice 90 with tumor area 4483
    Image saved: /home/magd6292/Documents/wentian_clone/MultiUnet/dataset_samples/ISLES_Single_image.png
    Label saved: /home/magd6292/Documents/wentian_clone/MultiUnet/dataset_samples/ISLES_Single_label.png

Processing WMH...
  Checking file 1: train_0_u.nii.gz
    Found better slice 37 with tumor area 1550
  Checking file 2: train_100_a.nii.gz
  Checking file 3: train_101_a.nii.gz
  Checking file 4: train_102_a.nii.gz
  Checking file 5: train_103_a.nii.gz
  ✓ Created images for WMH
    Axial slice 37 with tumor area 1550
    Image saved: /home/magd6292/Documents/wentian_clone/MultiUnet/dataset_samples/WMH_Single_image.png
    Label saved: /home/magd6292/Documents/wentian_clone/MultiUnet/dataset_samples/WMH_Single_label.png

Processing BRATS...
  Checking file 1: BR

In [41]:
# # Test the updated matching function for ISLES datasets
# print("Testing updated matching function...")
# print("=" * 50)

# for dataset in ['ISLES', 'ISLES2022']:
#     print(f"\n{dataset}:")
#     pairs = find_matching_files(dataset, data_folder)
#     print(f"  Found {len(pairs)} matching pairs")
#     if len(pairs) > 0:
#         print(f"  Example: {os.path.basename(pairs[0][0])} -> {os.path.basename(pairs[0][1])}")
#         if len(pairs) > 1:
#             print(f"  Example: {os.path.basename(pairs[1][0])} -> {os.path.basename(pairs[1][1])}")
#     print()


In [42]:
# def save_image_files(image_slice, label_slice, dataset_name, modality_name, slice_idx, output_folder):
#     """Save rotated image and label with black background for labels - all same size"""
    
#     # Create output folder if it doesn't exist
#     os.makedirs(output_folder, exist_ok=True)
    
#     if image_slice is not None:
#         # Rotate anticlockwise by 90 degrees
#         image_slice_rotated = np.rot90(image_slice, k=1)  # k=1 for 90° anticlockwise
#         label_slice_rotated = np.rot90(label_slice, k=1)
        
#         # Normalize image slice for better visualization
#         image_normalized = (image_slice_rotated - np.min(image_slice_rotated)) / (np.max(image_slice_rotated) - np.min(image_slice_rotated) + 1e-8)
        
#         # Create filename
#         filename_base = f"{dataset_name}_{modality_name}" if modality_name else dataset_name
        
#         # FIXED SIZE SETTINGS - ensures all images are exactly the same size
#         fig_size = (8, 8)  # Fixed figure size
#         dpi = 200          # Fixed DPI
        
#         # Save image only
#         fig, ax = plt.subplots(figsize=fig_size)
#         ax.imshow(image_normalized, cmap='gray', aspect='equal')
#         ax.set_xlim([0, image_normalized.shape[1]])
#         ax.set_ylim([image_normalized.shape[0], 0])  # Flip Y axis for proper orientation
#         ax.axis('off')
        
#         image_path = os.path.join(output_folder, f'{filename_base}_image.png')
#         plt.savefig(image_path, bbox_inches='tight', pad_inches=0, dpi=dpi, 
#                    facecolor='white', edgecolor='none')
#         plt.close()
        
#         # Save label only with black background and red tumor
#         fig, ax = plt.subplots(figsize=fig_size)
        
#         # Create a black background
#         black_background = np.zeros_like(label_slice_rotated)
#         ax.imshow(black_background, cmap='gray', aspect='equal', vmin=0, vmax=1)
        
#         # Overlay tumor in red
#         if np.sum(label_slice_rotated) > 0:
#             tumor_mask = label_slice_rotated > 0
#             ax.imshow(np.ma.masked_where(~tumor_mask, label_slice_rotated), 
#                      cmap='Reds', alpha=1.0, aspect='equal', vmin=0, vmax=1)
        
#         ax.set_xlim([0, label_slice_rotated.shape[1]])
#         ax.set_ylim([label_slice_rotated.shape[0], 0])  # Flip Y axis for proper orientation
#         ax.axis('off')
        
#         label_path = os.path.join(output_folder, f'{filename_base}_label.png')
#         plt.savefig(label_path, bbox_inches='tight', pad_inches=0, dpi=dpi, 
#                    facecolor='black', edgecolor='none')
#         plt.close()
        
#         return image_path, label_path
    
#     return None, None


In [43]:
# # FINAL PROCESSING: Create fixed-size images for ALL datasets including ISLES
# results_final = {}

# print('Creating FIXED-SIZE rotated images for ALL datasets and modalities...')
# print('=' * 70)

# # Include ALL datasets now that ISLES matching is fixed
# all_datasets = ['ISLES', 'BRATS', 'ISLES2022', 'TBI', 'ATLAS', 'MSSEG', 'TUMOUR2']

# for dataset in all_datasets:
#     print(f'\nProcessing {dataset}...')
    
#     pairs = find_matching_files(dataset, data_folder)
    
#     if len(pairs) == 0:
#         print(f'  No matching pairs found for {dataset}')
#         continue
    
#     best_result = None
#     max_tumor_area = 0
    
#     # Check first 5 files to find best tumor
#     for i, (image_path, label_path) in enumerate(pairs[:5]):
#         print(f'  Checking file {i+1}: {os.path.basename(image_path)}')
        
#         result = find_best_axial_tumor_slice_multimodal(image_path, label_path)
        
#         if len(result) == 5:
#             image_slices, modality_names, label_slice, slice_idx, tumor_area = result
            
#             if image_slices is not None and tumor_area > max_tumor_area:
#                 max_tumor_area = tumor_area
#                 best_result = (image_slices, modality_names, label_slice, slice_idx, tumor_area, image_path, label_path)
#                 print(f'    Found better slice {slice_idx} with tumor area {tumor_area}')
#                 print(f'    Modalities: {modality_names}')
    
#     if best_result is not None:
#         image_slices, modality_names, label_slice, slice_idx, tumor_area, image_path, label_path = best_result
        
#         dataset_results = {
#             'original_image': image_path,
#             'original_label': label_path,
#             'slice_index': slice_idx,
#             'tumor_area': tumor_area,
#             'modalities': {}
#         }
        
#         # Save FIXED-SIZE image for each modality
#         for mod_idx, (image_slice, modality_name) in enumerate(zip(image_slices, modality_names)):
#             print(f'  Creating FIXED-SIZE images for {dataset} - {modality_name}')
            
#             img_path, lbl_path = save_image_files(
#                 image_slice, label_slice, dataset, modality_name, slice_idx, output_folder
#             )
            
#             dataset_results['modalities'][modality_name] = {
#                 'image_png': img_path,
#                 'label_png': lbl_path
#             }
        
#         results_final[dataset] = dataset_results
        
#         print(f'  ✓ Created FIXED-SIZE images for {dataset}')
#         print(f'    Axial slice {slice_idx} with tumor area {tumor_area}')
#         print(f'    Created {len(modality_names)} modality images')
#     else:
#         print(f'  ✗ No suitable tumor found for {dataset}')

# print(f'\n' + '=' * 70)
# print(f'FIXED-SIZE IMAGE CREATION COMPLETE')
# print(f'=' * 70)
# print(f'Successfully created FIXED-SIZE images for {len(results_final)} datasets:')

# total_images = 0
# for dataset, info in results_final.items():
#     modalities = list(info['modalities'].keys())
#     total_images += len(modalities) * 2  # 2 files per modality (image + label)
#     slice_index = info['slice_index']
#     tumor_area = info['tumor_area']
#     print(f'  {dataset}: axial slice {slice_index} - tumor area: {tumor_area}')
#     print(f'    Modalities: {", ".join(modalities)}')

# print(f'\nTotal FIXED-SIZE image files created: {total_images}')
# print(f'All files saved to: {output_folder}')
# print('\nFiles created (ALL SAME SIZE):')
# for dataset, info in results_final.items():
#     for modality in info['modalities'].keys():
#         filename_base = f'{dataset}_{modality}'
#         print(f'  - {filename_base}_image.png (fixed-size rotated brain image)')
#         print(f'  - {filename_base}_label.png (fixed-size rotated tumor mask with black background)')


In [44]:
# def save_truly_fixed_size_images(image_slice, label_slice, dataset_name, modality_name, slice_idx, output_folder):
#     """Save images with EXACTLY the same pixel dimensions - no bbox_inches='tight'"""
    
#     # Create output folder if it doesn't exist
#     os.makedirs(output_folder, exist_ok=True)
    
#     if image_slice is not None:
#         # Rotate anticlockwise by 90 degrees
#         image_slice_rotated = np.rot90(image_slice, k=1)  # k=1 for 90° anticlockwise
#         label_slice_rotated = np.rot90(label_slice, k=1)
        
#         # Normalize image slice for better visualization
#         image_normalized = (image_slice_rotated - np.min(image_slice_rotated)) / (np.max(image_slice_rotated) - np.min(image_slice_rotated) + 1e-8)
        
#         # Create filename
#         filename_base = f"{dataset_name}_{modality_name}" if modality_name else dataset_name
        
#         # TRULY FIXED SIZE SETTINGS - NO bbox_inches='tight'!
#         fig_width = 8  # inches
#         fig_height = 8  # inches
#         dpi = 200      # Fixed DPI
#         # This will create exactly: 1600x1600 pixels for ALL images
        
#         # Save image only - FIXED SIZE
#         fig, ax = plt.subplots(figsize=(fig_width, fig_height))
#         ax.imshow(image_normalized, cmap='gray', aspect='equal')
#         ax.set_xlim([0, image_normalized.shape[1]])
#         ax.set_ylim([image_normalized.shape[0], 0])  # Flip Y axis for proper orientation
#         ax.axis('off')
        
#         # Remove all margins and padding
#         plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
        
#         image_path = os.path.join(output_folder, f'{filename_base}_image.png')
#         plt.savefig(image_path, dpi=dpi, facecolor='white', edgecolor='none')
#         plt.close()
        
#         # Save label only - FIXED SIZE with black background
#         fig, ax = plt.subplots(figsize=(fig_width, fig_height))
        
#         # Create a black background
#         black_background = np.zeros_like(label_slice_rotated)
#         ax.imshow(black_background, cmap='gray', aspect='equal', vmin=0, vmax=1)
        
#         # Overlay tumor in red
#         if np.sum(label_slice_rotated) > 0:
#             tumor_mask = label_slice_rotated > 0
#             ax.imshow(np.ma.masked_where(~tumor_mask, label_slice_rotated), 
#                      cmap='Reds', alpha=1.0, aspect='equal', vmin=0, vmax=1)
        
#         ax.set_xlim([0, label_slice_rotated.shape[1]])
#         ax.set_ylim([label_slice_rotated.shape[0], 0])  # Flip Y axis for proper orientation
#         ax.axis('off')
        
#         # Remove all margins and padding
#         plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
        
#         label_path = os.path.join(output_folder, f'{filename_base}_label.png')
#         plt.savefig(label_path, dpi=dpi, facecolor='black', edgecolor='none')
#         plt.close()
        
#         return image_path, label_path
    
#     return None, None


In [45]:
# # TRULY FIXED SIZE PROCESSING: All images will be EXACTLY 1600x1600 pixels
# results_truly_fixed = {}

# print('Creating TRULY FIXED-SIZE (1600x1600 pixels) images for ALL datasets...')
# print('=' * 75)

# # Include ALL datasets
# all_datasets = ['ISLES', 'BRATS', 'ISLES2022', 'TBI', 'ATLAS', 'MSSEG', 'TUMOUR2']

# for dataset in all_datasets:
#     print(f'\nProcessing {dataset}...')
    
#     pairs = find_matching_files(dataset, data_folder)
    
#     if len(pairs) == 0:
#         print(f'  No matching pairs found for {dataset}')
#         continue
    
#     best_result = None
#     max_tumor_area = 0
    
#     # Check first 5 files to find best tumor
#     for i, (image_path, label_path) in enumerate(pairs[:5]):
#         print(f'  Checking file {i+1}: {os.path.basename(image_path)}')
        
#         result = find_best_axial_tumor_slice_multimodal(image_path, label_path)
        
#         if len(result) == 5:
#             image_slices, modality_names, label_slice, slice_idx, tumor_area = result
            
#             if image_slices is not None and tumor_area > max_tumor_area:
#                 max_tumor_area = tumor_area
#                 best_result = (image_slices, modality_names, label_slice, slice_idx, tumor_area, image_path, label_path)
#                 print(f'    Found better slice {slice_idx} with tumor area {tumor_area}')
#                 print(f'    Modalities: {modality_names}')
    
#     if best_result is not None:
#         image_slices, modality_names, label_slice, slice_idx, tumor_area, image_path, label_path = best_result
        
#         dataset_results = {
#             'original_image': image_path,
#             'original_label': label_path,
#             'slice_index': slice_idx,
#             'tumor_area': tumor_area,
#             'modalities': {}
#         }
        
#         # Save TRULY FIXED-SIZE image for each modality
#         for mod_idx, (image_slice, modality_name) in enumerate(zip(image_slices, modality_names)):
#             print(f'  Creating 1600x1600 pixel images for {dataset} - {modality_name}')
            
#             img_path, lbl_path = save_truly_fixed_size_images(
#                 image_slice, label_slice, dataset, modality_name, slice_idx, output_folder
#             )
            
#             dataset_results['modalities'][modality_name] = {
#                 'image_png': img_path,
#                 'label_png': lbl_path
#             }
        
#         results_truly_fixed[dataset] = dataset_results
        
#         print(f'  ✓ Created 1600x1600 pixel images for {dataset}')
#         print(f'    Axial slice {slice_idx} with tumor area {tumor_area}')
#         print(f'    Created {len(modality_names)} modality images')
#     else:
#         print(f'  ✗ No suitable tumor found for {dataset}')

# print(f'\n' + '=' * 75)
# print(f'TRULY FIXED-SIZE (1600x1600 pixels) IMAGE CREATION COMPLETE')
# print(f'=' * 75)
# print(f'Successfully created images for {len(results_truly_fixed)} datasets:')

# total_images = 0
# for dataset, info in results_truly_fixed.items():
#     modalities = list(info['modalities'].keys())
#     total_images += len(modalities) * 2  # 2 files per modality (image + label)
#     slice_index = info['slice_index']
#     tumor_area = info['tumor_area']
#     print(f'  {dataset}: axial slice {slice_index} - tumor area: {tumor_area}')
#     print(f'    Modalities: {", ".join(modalities)}')

# print(f'\nTotal image files created: {total_images}')
# print(f'All files saved to: {output_folder}')
# print('\nAll PNG files are now EXACTLY 1600x1600 pixels!')
# print('\nFiles created (ALL EXACTLY 1600x1600 pixels):')
# for dataset, info in results_truly_fixed.items():
#     for modality in info['modalities'].keys():
#         filename_base = f'{dataset}_{modality}'
#         print(f'  - {filename_base}_image.png (1600x1600 pixels)')
#         print(f'  - {filename_base}_label.png (1600x1600 pixels)')


In [46]:
# def save_absolutely_same_size_images(image_slice, label_slice, dataset_name, modality_name, slice_idx, output_folder):
#     """Save images that are ABSOLUTELY the same size by FORCING resize to fixed dimensions"""
    
#     # Create output folder if it doesn't exist
#     os.makedirs(output_folder, exist_ok=True)
    
#     if image_slice is not None:
#         # Rotate anticlockwise by 90 degrees
#         image_slice_rotated = np.rot90(image_slice, k=1)  # k=1 for 90° anticlockwise
#         label_slice_rotated = np.rot90(label_slice, k=1)
        
#         # Normalize image slice for better visualization
#         image_normalized = (image_slice_rotated - np.min(image_slice_rotated)) / (np.max(image_slice_rotated) - np.min(image_slice_rotated) + 1e-8)
        
#         # Create filename
#         filename_base = f"{dataset_name}_{modality_name}" if modality_name else dataset_name
        
#         # ABSOLUTE FIXED DIMENSIONS - FORCE RESIZE TO EXACT SIZE
#         TARGET_SIZE = 512  # All images will be EXACTLY 512x512 pixels
        
#         # Resize image data to exact target size using scipy
#         from scipy.ndimage import zoom
        
#         # Calculate zoom factors to get exact target size
#         zoom_factor_y = TARGET_SIZE / image_normalized.shape[0]
#         zoom_factor_x = TARGET_SIZE / image_normalized.shape[1]
        
#         # Resize image and label to EXACTLY the target size
#         image_resized = zoom(image_normalized, [zoom_factor_y, zoom_factor_x], order=1)
#         label_resized = zoom(label_slice_rotated, [zoom_factor_y, zoom_factor_x], order=0)  # Nearest neighbor for labels
        
#         # Ensure exact size (in case of rounding errors)
#         image_resized = image_resized[:TARGET_SIZE, :TARGET_SIZE]
#         label_resized = label_resized[:TARGET_SIZE, :TARGET_SIZE]
        
#         # Pad if needed to ensure exact size
#         if image_resized.shape[0] < TARGET_SIZE or image_resized.shape[1] < TARGET_SIZE:
#             padded_image = np.zeros((TARGET_SIZE, TARGET_SIZE))
#             padded_label = np.zeros((TARGET_SIZE, TARGET_SIZE))
            
#             padded_image[:image_resized.shape[0], :image_resized.shape[1]] = image_resized
#             padded_label[:label_resized.shape[0], :label_resized.shape[1]] = label_resized
            
#             image_resized = padded_image
#             label_resized = padded_label
        
#         # Save image - EXACT SIZE using matplotlib with no margins
#         fig, ax = plt.subplots(figsize=(5.12, 5.12))  # 5.12 inches at 100 DPI = 512 pixels
#         ax.imshow(image_resized, cmap='gray', aspect='equal', extent=[0, TARGET_SIZE, TARGET_SIZE, 0])
#         ax.set_xlim(0, TARGET_SIZE)
#         ax.set_ylim(TARGET_SIZE, 0)
#         ax.axis('off')
        
#         # Remove all margins completely
#         plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
        
#         image_path = os.path.join(output_folder, f'{filename_base}_image.png')
#         plt.savefig(image_path, dpi=100, facecolor='white', edgecolor='none', 
#                    bbox_inches='tight', pad_inches=0)
#         plt.close()
        
#         # Save label - EXACT SIZE with black background
#         fig, ax = plt.subplots(figsize=(5.12, 5.12))  # 5.12 inches at 100 DPI = 512 pixels
        
#         # Create black background
#         black_bg = np.zeros((TARGET_SIZE, TARGET_SIZE))
#         ax.imshow(black_bg, cmap='gray', aspect='equal', extent=[0, TARGET_SIZE, TARGET_SIZE, 0])
        
#         # Overlay tumor in red
#         if np.sum(label_resized) > 0:
#             tumor_mask = label_resized > 0
#             masked_label = np.ma.masked_where(~tumor_mask, label_resized)
#             ax.imshow(masked_label, cmap='Reds', alpha=1.0, aspect='equal', 
#                      extent=[0, TARGET_SIZE, TARGET_SIZE, 0], vmin=0, vmax=1)
        
#         ax.set_xlim(0, TARGET_SIZE)
#         ax.set_ylim(TARGET_SIZE, 0)
#         ax.axis('off')
        
#         # Remove all margins completely
#         plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
        
#         label_path = os.path.join(output_folder, f'{filename_base}_label.png')
#         plt.savefig(label_path, dpi=100, facecolor='black', edgecolor='none',
#                    bbox_inches='tight', pad_inches=0)
#         plt.close()
        
#         return image_path, label_path
    
#     return None, None


In [47]:
# def save_uniform_size_images_no_crop(image_slice, label_slice, dataset_name, modality_name, slice_idx, output_folder):
#     """Save images with EXACTLY the same dimensions - resize but NEVER crop the brain"""
    
#     # Create output folder if it doesn't exist
#     os.makedirs(output_folder, exist_ok=True)
    
#     if image_slice is not None:
#         # Rotate anticlockwise by 90 degrees
#         image_slice_rotated = np.rot90(image_slice, k=1)  # k=1 for 90° anticlockwise
#         label_slice_rotated = np.rot90(label_slice, k=1)
        
#         # Normalize image slice for better visualization
#         image_normalized = (image_slice_rotated - np.min(image_slice_rotated)) / (np.max(image_slice_rotated) - np.min(image_slice_rotated) + 1e-8)
        
#         # Create filename
#         filename_base = f"{dataset_name}_{modality_name}" if modality_name else dataset_name
        
#         # TARGET DIMENSIONS - All images will be EXACTLY this size
#         TARGET_WIDTH = 512
#         TARGET_HEIGHT = 512
        
#         # Resize using scipy.ndimage to preserve full brain content
#         from scipy.ndimage import zoom
        
#         # Calculate zoom factors
#         zoom_factor_y = TARGET_HEIGHT / image_normalized.shape[0]
#         zoom_factor_x = TARGET_WIDTH / image_normalized.shape[1]
        
#         # Use the SAME zoom factor for both dimensions to maintain aspect ratio
#         # Use the smaller zoom factor to ensure the entire brain fits
#         zoom_factor = min(zoom_factor_y, zoom_factor_x)
        
#         # Resize with uniform scaling (no distortion)
#         image_resized = zoom(image_normalized, [zoom_factor, zoom_factor], order=1)
#         label_resized = zoom(label_slice_rotated, [zoom_factor, zoom_factor], order=0)  # Nearest neighbor for labels
        
#         # Create target-sized arrays and center the resized image
#         final_image = np.zeros((TARGET_HEIGHT, TARGET_WIDTH))
#         final_label = np.zeros((TARGET_HEIGHT, TARGET_WIDTH))
        
#         # Calculate centering offsets
#         y_offset = (TARGET_HEIGHT - image_resized.shape[0]) // 2
#         x_offset = (TARGET_WIDTH - image_resized.shape[1]) // 2
        
#         # Place the resized image in the center
#         final_image[y_offset:y_offset+image_resized.shape[0], 
#                    x_offset:x_offset+image_resized.shape[1]] = image_resized
#         final_label[y_offset:y_offset+label_resized.shape[0], 
#                    x_offset:x_offset+label_resized.shape[1]] = label_resized
        
#         # Save image - EXACT DIMENSIONS using direct array creation
#         fig, ax = plt.subplots(figsize=(5.12, 5.12))  # 5.12 inches at 100 DPI = 512 pixels
#         ax.imshow(final_image, cmap='gray', aspect='equal')
#         ax.set_xlim(0, TARGET_WIDTH)
#         ax.set_ylim(TARGET_HEIGHT, 0)
#         ax.axis('off')
        
#         # Remove ALL margins and padding
#         plt.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0, hspace=0)
        
#         image_path = os.path.join(output_folder, f'{filename_base}_image.png')
#         plt.savefig(image_path, dpi=100, facecolor='black', edgecolor='none')
#         plt.close()
        
#         # Save label - EXACT DIMENSIONS with black background
#         fig, ax = plt.subplots(figsize=(5.12, 5.12))  # 5.12 inches at 100 DPI = 512 pixels
        
#         # Black background
#         ax.imshow(np.zeros((TARGET_HEIGHT, TARGET_WIDTH)), cmap='gray', aspect='equal')
        
#         # Overlay tumor in red
#         if np.sum(final_label) > 0:
#             tumor_mask = final_label > 0
#             masked_label = np.ma.masked_where(~tumor_mask, final_label)
#             ax.imshow(masked_label, cmap='Reds', alpha=1.0, aspect='equal', vmin=0, vmax=1)
        
#         ax.set_xlim(0, TARGET_WIDTH)
#         ax.set_ylim(TARGET_HEIGHT, 0)
#         ax.axis('off')
        
#         # Remove ALL margins and padding
#         plt.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0, hspace=0)
        
#         label_path = os.path.join(output_folder, f'{filename_base}_label.png')
#         plt.savefig(label_path, dpi=100, facecolor='black', edgecolor='none')
#         plt.close()
        
#         return image_path, label_path
    
#     return None, None


In [48]:
# def find_best_axial_tumor_slice_with_config(image_path, label_path):
#     """Find the best axial slice with proper modality names from config and patient info"""
#     try:
#         # Load the image and label
#         image_nii = nib.load(image_path)
#         label_nii = nib.load(label_path)
        
#         image_data = image_nii.get_fdata()
#         label_data = label_nii.get_fdata()
        
#         # Handle different dimensions for labels
#         if len(label_data.shape) == 4:
#             label_data = label_data[:, :, :, 0]  # Take first channel
        
#         # Focus on axial slices (z-axis, typically axis 2)
#         # Find slices with good tumor visibility in the middle region of the brain
#         z_slices = label_data.shape[2]
        
#         # Look in the middle 60% of slices (avoid top and bottom)
#         start_slice = int(z_slices * 0.2)
#         end_slice = int(z_slices * 0.8)
        
#         best_slice_idx = start_slice
#         max_score = 0
        
#         for i in range(start_slice, end_slice):
#             slice_data = label_data[:, :, i]
#             tumor_area = np.sum(slice_data > 0)
            
#             # Score based on tumor area and position (prefer middle slices)
#             middle_bonus = 1.0 - abs(i - z_slices/2) / (z_slices/2)  # Higher score for middle slices
#             score = tumor_area * (1 + 0.5 * middle_bonus)  # Bonus for being in middle
            
#             if score > max_score and tumor_area > 100:  # Minimum tumor size threshold
#                 max_score = score
#                 best_slice_idx = i
        
#         # Extract the best axial slice for all modalities
#         label_slice = label_data[:, :, best_slice_idx]
#         tumor_area = np.sum(label_slice > 0)
        
#         # Get dataset name from path for proper modality naming
#         dataset_name = None
#         for ds in ['BRATS', 'ATLAS', 'MSSEG', 'ISLES', 'TBI', 'ISLES2022', 'TUMOUR2']:
#             if ds in image_path:
#                 dataset_name = ds
#                 break
        
#         # Get patient/file name for verification
#         patient_name = os.path.basename(image_path).replace('.nii.gz', '')
        
#         # Handle multi-modal images (4D) or single modal (3D)
#         if len(image_data.shape) == 4:
#             # Multi-modal: return all modalities
#             image_slices = []
#             num_modalities = image_data.shape[3]
            
#             # Define modalities based on config.py - exact order as they are stacked
#             dataset_modalities = {
#                 'BRATS': ["FLAIR", "T1", "T1c", "T2"],
#                 'ATLAS': ["T1"],
#                 'MSSEG': ['FLAIR', "T1", "T1c", "T2", "PD"],
#                 'ISLES': ["FLAIR", "T1", "T2", "DWI"],
#                 'TBI': ["FLAIR", "T1", "T2", "SWI"],
#                 'ISLES2022': ['ADC', 'DWI', 'FLAIR'],  # Corrected: ADC, DWI, FLAIR (not T1, T2)
#                 'TUMOUR2': ['T1']
#             }
            
#             if dataset_name and dataset_name in dataset_modalities:
#                 config_modalities = dataset_modalities[dataset_name]
#                 # Use the config modalities, but only up to the number we actually have
#                 modality_names = config_modalities[:num_modalities]
#                 # If we have more modalities than expected, fill with generic names
#                 if num_modalities > len(config_modalities):
#                     for i in range(len(config_modalities), num_modalities):
#                         modality_names.append(f'Modality_{i+1}')
#             else:
#                 # Fallback to generic names
#                 modality_names = [f'Modality_{i+1}' for i in range(num_modalities)]
            
#             for mod_idx in range(num_modalities):
#                 image_slice = image_data[:, :, best_slice_idx, mod_idx]
#                 image_slices.append(image_slice)
                
#             return image_slices, modality_names, label_slice, best_slice_idx, tumor_area, patient_name
#         else:
#             # Single modal: return as list for consistency
#             image_slice = image_data[:, :, best_slice_idx]
#             return [image_slice], ['Single'], label_slice, best_slice_idx, tumor_area, patient_name
        
#     except Exception as e:
#         print(f"Error processing {image_path}: {str(e)}")
#         return None, None, None, 0, 0, None


In [49]:
# # FINAL CORRECTED: BLACK PADDING with CONFIG-BASED MODALITIES and PATIENT INFO
# results_final_corrected = {}

# print('Creating images with PROPER MODALITY NAMES from config and BLACK PADDING...')
# print('=' * 80)

# # Include ALL datasets
# all_datasets = ['ISLES', 'BRATS', 'ISLES2022', 'TBI', 'ATLAS', 'MSSEG', 'TUMOUR2']

# for dataset in all_datasets:
#     print(f'\n🔍 Processing {dataset}...')
    
#     pairs = find_matching_files(dataset, data_folder)
    
#     if len(pairs) == 0:
#         print(f'  ❌ No matching pairs found for {dataset}')
#         continue
    
#     best_result = None
#     max_tumor_area = 0
    
#     # Check first 5 files to find best tumor
#     for i, (image_path, label_path) in enumerate(pairs[:5]):
#         print(f'  📁 Checking file {i+1}: {os.path.basename(image_path)}')
        
#         result = find_best_axial_tumor_slice_with_config(image_path, label_path)
        
#         if len(result) == 6 and result[0] is not None:  # Updated for 6 return values
#             image_slices, modality_names, label_slice, slice_idx, tumor_area, patient_name = result
            
#             if image_slices is not None and tumor_area > max_tumor_area:
#                 max_tumor_area = tumor_area
#                 best_result = (image_slices, modality_names, label_slice, slice_idx, tumor_area, patient_name, image_path, label_path)
#                 print(f'    ✅ Found better slice {slice_idx} with tumor area {tumor_area}')
#                 print(f'    👤 Patient: {patient_name}')
#                 print(f'    🧠 Modalities: {modality_names}')
    
#     if best_result is not None:
#         image_slices, modality_names, label_slice, slice_idx, tumor_area, patient_name, image_path, label_path = best_result
        
#         dataset_results = {
#             'original_image': image_path,
#             'original_label': label_path,
#             'patient_name': patient_name,
#             'slice_index': slice_idx,
#             'tumor_area': tumor_area,
#             'modalities': {}
#         }
        
#         # Save BLACK PADDING images for each modality with correct names
#         for mod_idx, (image_slice, modality_name) in enumerate(zip(image_slices, modality_names)):
#             print(f'  🎨 Creating {dataset} - {modality_name} (Patient: {patient_name})')
            
#             img_path, lbl_path = save_uniform_size_images_no_crop(
#                 image_slice, label_slice, dataset, modality_name, slice_idx, output_folder
#             )
            
#             dataset_results['modalities'][modality_name] = {
#                 'image_png': img_path,
#                 'label_png': lbl_path
#             }
        
#         results_final_corrected[dataset] = dataset_results
        
#         print(f'  ✅ Created images for {dataset}')
#         print(f'    👤 Patient: {patient_name}')
#         print(f'    🎯 Axial slice {slice_idx} with tumor area {tumor_area}')
#         print(f'    📊 Created {len(modality_names)} modality images: {", ".join(modality_names)}')
#     else:
#         print(f'  ❌ No suitable tumor found for {dataset}')

# print(f'\n' + '🎉' + '=' * 78 + '🎉')
# print(f'✅ CORRECTED IMAGE CREATION COMPLETE WITH PROPER MODALITIES')
# print(f'🎉' + '=' * 78 + '🎉')
# print(f'Successfully created images for {len(results_final_corrected)} datasets:')

# total_images = 0
# for dataset, info in results_final_corrected.items():
#     modalities = list(info['modalities'].keys())
#     total_images += len(modalities) * 2  # 2 files per modality (image + label)
#     slice_index = info['slice_index']
#     tumor_area = info['tumor_area']
#     patient_name = info['patient_name']
#     print(f'  📊 {dataset}: Patient {patient_name}, slice {slice_index}, tumor area: {tumor_area}')
#     print(f'      🧠 Modalities: {", ".join(modalities)}')

# print(f'\n📈 Total image files created: {total_images}')
# print(f'📂 All files saved to: {output_folder}')
# print(f'🎨 All PNG files have BLACK PADDING and CORRECT modality names from config!')
# print('\n📋 Files created:')
# for dataset, info in results_final_corrected.items():
#     patient = info['patient_name']
#     for modality in info['modalities'].keys():
#         filename_base = f'{dataset}_{modality}'
#         print(f'  📄 {filename_base}_image.png (Patient: {patient})')
#         print(f'  📄 {filename_base}_label.png (Patient: {patient})')


In [50]:
# # Quick test to verify ISLES2022 modalities are correctly detected
# print("🔍 Testing ISLES2022 modality detection...")
# print("=" * 50)

# isles2022_pairs = find_matching_files('ISLES2022', data_folder)
# if isles2022_pairs:
#     test_image, test_label = isles2022_pairs[0]
#     print(f"📁 Testing with: {os.path.basename(test_image)}")
    
#     # Test with the corrected function
#     result = find_best_axial_tumor_slice_with_config(test_image, test_label)
#     if len(result) == 6 and result[0] is not None:
#         image_slices, modality_names, label_slice, slice_idx, tumor_area, patient_name = result
#         print(f"✅ Corrected function results:")
#         print(f"   👤 Patient: {patient_name}")
#         print(f"   🧠 Modalities: {modality_names}")
#         print(f"   📊 Number of modalities: {len(modality_names)}")
#         print(f"   🎯 Expected: ['ADC', 'DWI', 'FLAIR']")
        
#         # Check if correct
#         expected = ['ADC', 'DWI', 'FLAIR']
#         if modality_names == expected:
#             print("   ✅ CORRECT! Modalities match config.py")
#         else:
#             print(f"   ❌ MISMATCH! Got {modality_names}, expected {expected}")
    
#     # Test with old function for comparison
#     result_old = find_best_axial_tumor_slice_multimodal(test_image, test_label)
#     if len(result_old) == 5 and result_old[0] is not None:
#         image_slices_old, modality_names_old, label_slice_old, slice_idx_old, tumor_area_old = result_old
#         print(f"\n❌ Old function results (INCORRECT):")
#         print(f"   🧠 Modalities: {modality_names_old}")
#         print(f"   📊 Number of modalities: {len(modality_names_old)}")
# else:
#     print("❌ No ISLES2022 pairs found for testing")


In [51]:
# FINAL PROCESSING: Correct modalities with black padding - ALL DATASETS
print("🎯 FINAL PROCESSING: Creating images with CORRECT modalities from config...")
print("=" * 80)

final_results = {}
all_datasets = ['ISLES', 'BRATS', 'ISLES2022', 'TBI', 'ATLAS', 'MSSEG', 'TUMOUR2','WMH']

for dataset in all_datasets:
    print(f'\n🔍 Processing {dataset}...')
    
    pairs = find_matching_files(dataset, data_folder)
    
    if len(pairs) == 0:
        print(f'  ❌ No matching pairs found for {dataset}')
        continue
    
    best_result = None
    max_tumor_area = 0
    
    # Check first 5 files to find best tumor
    for i, (image_path, label_path) in enumerate(pairs[:5]):
        print(f'  📁 Checking file {i+1}: {os.path.basename(image_path)}')
        
        # Use the CORRECTED function with proper modality names
        result = find_best_axial_tumor_slice_with_config(image_path, label_path)
        
        if len(result) == 6 and result[0] is not None:
            image_slices, modality_names, label_slice, slice_idx, tumor_area, patient_name = result
            
            if image_slices is not None and tumor_area > max_tumor_area:
                max_tumor_area = tumor_area
                best_result = (image_slices, modality_names, label_slice, slice_idx, tumor_area, patient_name, image_path, label_path)
                print(f'    ✅ Found better slice {slice_idx} with tumor area {tumor_area}')
                print(f'    👤 Patient: {patient_name}')
                print(f'    🧠 Modalities: {modality_names}')
    
    if best_result is not None:
        image_slices, modality_names, label_slice, slice_idx, tumor_area, patient_name, image_path, label_path = best_result
        
        dataset_results = {
            'original_image': image_path,
            'original_label': label_path,
            'patient_name': patient_name,
            'slice_index': slice_idx,
            'tumor_area': tumor_area,
            'modalities': {}
        }
        
        # Save images with CORRECT modality names and black padding
        for mod_idx, (image_slice, modality_name) in enumerate(zip(image_slices, modality_names)):
            print(f'  🎨 Creating {dataset} - {modality_name} (Patient: {patient_name})')
            
            img_path, lbl_path = save_uniform_size_images_no_crop(
                image_slice, label_slice, dataset, modality_name, slice_idx, output_folder
            )
            
            dataset_results['modalities'][modality_name] = {
                'image_png': img_path,
                'label_png': lbl_path
            }
        
        final_results[dataset] = dataset_results
        
        print(f'  ✅ Created images for {dataset}')
        print(f'    👤 Patient: {patient_name}')
        print(f'    🎯 Axial slice {slice_idx} with tumor area {tumor_area}')
        print(f'    📊 Created {len(modality_names)} modality images: {", ".join(modality_names)}')
    else:
        print(f'  ❌ No suitable tumor found for {dataset}')

print(f'\n' + '🎉' + '=' * 78 + '🎉')
print(f'✅ FINAL PROCESSING COMPLETE - ALL MODALITIES CORRECT!')
print(f'🎉' + '=' * 78 + '🎉')

# Summary with verification of correct modalities
print(f'Successfully created images for {len(final_results)} datasets:')
print()

# Expected modalities from config
expected_modalities = {
    'BRATS': ["FLAIR", "T1", "T1c", "T2"],
    'ATLAS': ["T1"],
    'MSSEG': ['FLAIR', "T1", "T1c", "T2", "PD"],
    'ISLES': ["FLAIR", "T1", "T2", "DWI"],
    'TBI': ["FLAIR", "T1", "T2", "SWI"],
    'ISLES2022': ['ADC', 'DWI', 'FLAIR'],  # ⭐ CORRECTED!
    'TUMOUR2': ['T1']
}

total_images = 0
for dataset, info in final_results.items():
    modalities = list(info['modalities'].keys())
    total_images += len(modalities) * 2
    patient_name = info['patient_name']
    slice_index = info['slice_index']
    tumor_area = info['tumor_area']
    
    # Check if modalities match expected
    expected = expected_modalities.get(dataset, [])
    if modalities == expected:
        status = "✅ CORRECT"
    else:
        status = "❌ MISMATCH"
    
    print(f'  📊 {dataset}: Patient {patient_name}, slice {slice_index}, tumor area: {tumor_area}')
    print(f'      🧠 Modalities: {", ".join(modalities)} {status}')
    if dataset == 'ISLES2022':
        print(f'      🎯 ISLES2022 now correctly shows: ADC, DWI, FLAIR (not T1, T2!)')

print(f'\n📈 Total image files created: {total_images}')
print(f'📂 All files saved to: {output_folder}')
print(f'🎨 All PNG files have BLACK PADDING and CORRECT modality names!')


🎯 FINAL PROCESSING: Creating images with CORRECT modalities from config...

🔍 Processing ISLES...
  🔄 Using ORDER-BASED matching for ISLES
  ✅ Matched 28 pairs by order
  📁 Example: 02_image.nii.gz -> 02_label.nii.gz
  📁 Checking file 1: 02_image.nii.gz
    ✅ Found better slice 78 with tumor area 862
    👤 Patient: 02_image
    🧠 Modalities: ['FLAIR', 'T1', 'T2', 'DWI']
  📁 Checking file 2: 03_image.nii.gz
  📁 Checking file 3: 04_image.nii.gz
    ✅ Found better slice 90 with tumor area 4483
    👤 Patient: 04_image
    🧠 Modalities: ['FLAIR', 'T1', 'T2', 'DWI']
  📁 Checking file 4: 06_image.nii.gz
  📁 Checking file 5: 07_image.nii.gz
  🎨 Creating ISLES - FLAIR (Patient: 04_image)
  🎨 Creating ISLES - T1 (Patient: 04_image)
  🎨 Creating ISLES - T2 (Patient: 04_image)
  🎨 Creating ISLES - DWI (Patient: 04_image)
  ✅ Created images for ISLES
    👤 Patient: 04_image
    🎯 Axial slice 90 with tumor area 4483
    📊 Created 4 modality images: FLAIR, T1, T2, DWI

🔍 Processing BRATS...
  📁 Check

In [52]:
# # Test the CORRECTED order-based matching for ISLES and ISLES2022
# print("🔍 Testing CORRECTED order-based matching...")
# print("=" * 60)

# for dataset in ['ISLES', 'ISLES2022']:
#     print(f"\n📂 Testing {dataset}:")
#     pairs = find_matching_files(dataset, data_folder)
#     print(f"   Found {len(pairs)} matching pairs")
    
#     if len(pairs) > 0:
#         print(f"   First 3 examples:")
#         for i, (img, lbl) in enumerate(pairs[:3]):
#             print(f"   {i+1}. {os.path.basename(img)} -> {os.path.basename(lbl)}")
        
#         # Test modality detection with corrected function
#         test_img, test_lbl = pairs[0]
#         result = find_best_axial_tumor_slice_with_config(test_img, test_lbl)
#         if len(result) == 6 and result[0] is not None:
#             image_slices, modality_names, label_slice, slice_idx, tumor_area, patient_name = result
#             print(f"   🧠 Detected modalities: {modality_names}")
            
#             # Check against config
#             expected_modalities = {
#                 'ISLES': ["FLAIR", "T1", "T2", "DWI"],
#                 'ISLES2022': ['ADC', 'DWI', 'FLAIR']
#             }
#             expected = expected_modalities[dataset]
#             if modality_names == expected:
#                 print(f"   ✅ CORRECT! Matches config: {expected}")
#             else:
#                 print(f"   ❌ MISMATCH! Got {modality_names}, expected {expected}")
#     else:
#         print(f"   ❌ No pairs found!")

# print(f"\n" + "=" * 60)
# print("Order-based matching test complete!")


In [53]:
# # 🎯 FINAL CORRECTED PROCESSING: Order-based matching + Correct modalities
# print("🎯 FINAL PROCESSING: Order-based matching with CORRECT modalities from config...")
# print("=" * 80)

# final_corrected_results = {}
# all_datasets = ['ISLES', 'BRATS', 'ISLES2022', 'TBI', 'ATLAS', 'MSSEG', 'TUMOUR2']

# # Expected modalities from config.py (EXACT order as stacked in files)
# config_modalities = {
#     'BRATS': ["FLAIR", "T1", "T1c", "T2"],
#     'ATLAS': ["T1"],
#     'MSSEG': ['FLAIR', "T1", "T1c", "T2", "PD"],
#     'ISLES': ["FLAIR", "T1", "T2", "DWI"],  # Order as stacked in 4D files
#     'TBI': ["FLAIR", "T1", "T2", "SWI"],
#     'ISLES2022': ['ADC', 'DWI', 'FLAIR'],  # ⭐ CORRECTED! ADC, DWI, FLAIR (not T1, T2)
#     'TUMOUR2': ['T1']
# }

# for dataset in all_datasets:
#     print(f'\n🔍 Processing {dataset}...')
#     expected_mods = config_modalities.get(dataset, [])
#     print(f'   🎯 Expected modalities from config: {expected_mods}')
    
#     pairs = find_matching_files(dataset, data_folder)
    
#     if len(pairs) == 0:
#         print(f'  ❌ No matching pairs found for {dataset}')
#         continue
    
#     best_result = None
#     max_tumor_area = 0
    
#     # Check first 5 files to find best tumor
#     for i, (image_path, label_path) in enumerate(pairs[:5]):
#         print(f'  📁 Checking file {i+1}: {os.path.basename(image_path)}')
        
#         # Use the CORRECTED function with proper modality names
#         result = find_best_axial_tumor_slice_with_config(image_path, label_path)
        
#         if len(result) == 6 and result[0] is not None:
#             image_slices, modality_names, label_slice, slice_idx, tumor_area, patient_name = result
            
#             if image_slices is not None and tumor_area > max_tumor_area:
#                 max_tumor_area = tumor_area
#                 best_result = (image_slices, modality_names, label_slice, slice_idx, tumor_area, patient_name, image_path, label_path)
#                 print(f'    ✅ Found better slice {slice_idx} with tumor area {tumor_area}')
#                 print(f'    👤 Patient: {patient_name}')
#                 print(f'    🧠 Detected modalities: {modality_names}')
                
#                 # Verify against config
#                 if modality_names == expected_mods:
#                     print(f'    ✅ CORRECT! Matches config')
#                 else:
#                     print(f'    ⚠️  Expected {expected_mods}, got {modality_names}')
    
#     if best_result is not None:
#         image_slices, modality_names, label_slice, slice_idx, tumor_area, patient_name, image_path, label_path = best_result
        
#         dataset_results = {
#             'original_image': image_path,
#             'original_label': label_path,
#             'patient_name': patient_name,
#             'slice_index': slice_idx,
#             'tumor_area': tumor_area,
#             'expected_modalities': expected_mods,
#             'detected_modalities': modality_names,
#             'modalities': {}
#         }
        
#         # Save images with CORRECT modality names and black padding
#         for mod_idx, (image_slice, modality_name) in enumerate(zip(image_slices, modality_names)):
#             print(f'  🎨 Creating {dataset} - {modality_name} (Patient: {patient_name})')
            
#             img_path, lbl_path = save_uniform_size_images_no_crop(
#                 image_slice, label_slice, dataset, modality_name, slice_idx, output_folder
#             )
            
#             dataset_results['modalities'][modality_name] = {
#                 'image_png': img_path,
#                 'label_png': lbl_path
#             }
        
#         final_corrected_results[dataset] = dataset_results
        
#         print(f'  ✅ Created images for {dataset}')
#         print(f'    👤 Patient: {patient_name}')
#         print(f'    🎯 Axial slice {slice_idx} with tumor area {tumor_area}')
#         print(f'    📊 Created {len(modality_names)} modality images: {", ".join(modality_names)}')
#     else:
#         print(f'  ❌ No suitable tumor found for {dataset}')

# print(f'\n' + '🎉' + '=' * 78 + '🎉')
# print(f'✅ FINAL CORRECTED PROCESSING COMPLETE!')
# print(f'🎉' + '=' * 78 + '🎉')

# # Summary with verification
# print(f'Successfully created images for {len(final_corrected_results)} datasets:')
# print()

# total_images = 0
# all_correct = True

# for dataset, info in final_corrected_results.items():
#     detected_mods = info['detected_modalities']
#     expected_mods = info['expected_modalities']
#     total_images += len(detected_mods) * 2
#     patient_name = info['patient_name']
#     slice_index = info['slice_index']
#     tumor_area = info['tumor_area']
    
#     # Check if modalities match expected
#     if detected_mods == expected_mods:
#         status = "✅ CORRECT"
#     else:
#         status = "❌ MISMATCH"
#         all_correct = False
    
#     print(f'  📊 {dataset}: Patient {patient_name}, slice {slice_index}, tumor area: {tumor_area}')
#     print(f'      🧠 Modalities: {", ".join(detected_mods)} {status}')
#     if dataset in ['ISLES2022']:
#         print(f'      🎯 {dataset} correctly shows: {", ".join(detected_mods)}')

# print(f'\n📈 Total image files created: {total_images}')
# print(f'📂 All files saved to: {output_folder}')

# if all_correct:
#     print(f'🎉 ALL MODALITIES ARE CORRECTLY LABELED ACCORDING TO CONFIG!')
# else:
#     print(f'⚠️  Some modalities may need adjustment')

# print(f'🎨 All PNG files have BLACK PADDING and are 512x512 pixels!')


In [54]:
# # FINAL CORRECTED: BLACK PADDING - All images exactly 512x512 pixels with BLACK surroundings
# results_black_padding = {}

# print('Creating UNIFORM SIZE (512x512) images with BLACK PADDING...')
# print('=' * 60)

# # Include ALL datasets
# all_datasets = ['ISLES', 'BRATS', 'ISLES2022', 'TBI', 'ATLAS', 'MSSEG', 'TUMOUR2']

# for dataset in all_datasets:
#     print(f'\nProcessing {dataset}...')
    
#     pairs = find_matching_files(dataset, data_folder)
    
#     if len(pairs) == 0:
#         print(f'  No matching pairs found for {dataset}')
#         continue
    
#     best_result = None
#     max_tumor_area = 0
    
#     # Check first 5 files to find best tumor
#     for i, (image_path, label_path) in enumerate(pairs[:5]):
#         print(f'  Checking file {i+1}: {os.path.basename(image_path)}')
        
#         result = find_best_axial_tumor_slice_multimodal(image_path, label_path)
        
#         if len(result) == 5:
#             image_slices, modality_names, label_slice, slice_idx, tumor_area = result
            
#             if image_slices is not None and tumor_area > max_tumor_area:
#                 max_tumor_area = tumor_area
#                 best_result = (image_slices, modality_names, label_slice, slice_idx, tumor_area, image_path, label_path)
#                 print(f'    Found better slice {slice_idx} with tumor area {tumor_area}')
#                 print(f'    Modalities: {modality_names}')
    
#     if best_result is not None:
#         image_slices, modality_names, label_slice, slice_idx, tumor_area, image_path, label_path = best_result
        
#         dataset_results = {
#             'original_image': image_path,
#             'original_label': label_path,
#             'slice_index': slice_idx,
#             'tumor_area': tumor_area,
#             'modalities': {}
#         }
        
#         # Save BLACK PADDING images for each modality
#         for mod_idx, (image_slice, modality_name) in enumerate(zip(image_slices, modality_names)):
#             print(f'  Creating BLACK PADDING 512x512 images for {dataset} - {modality_name}')
            
#             img_path, lbl_path = save_uniform_size_images_no_crop(
#                 image_slice, label_slice, dataset, modality_name, slice_idx, output_folder
#             )
            
#             dataset_results['modalities'][modality_name] = {
#                 'image_png': img_path,
#                 'label_png': lbl_path
#             }
        
#         results_black_padding[dataset] = dataset_results
        
#         print(f'  ✓ Created BLACK PADDING images for {dataset}')
#         print(f'    Axial slice {slice_idx} with tumor area {tumor_area}')
#         print(f'    Created {len(modality_names)} modality images')
#     else:
#         print(f'  ✗ No suitable tumor found for {dataset}')

# print(f'\n' + '=' * 60)
# print(f'BLACK PADDING IMAGE CREATION COMPLETE')
# print(f'=' * 60)
# print(f'Successfully created images for {len(results_black_padding)} datasets:')

# total_images = 0
# for dataset, info in results_black_padding.items():
#     modalities = list(info['modalities'].keys())
#     total_images += len(modalities) * 2  # 2 files per modality (image + label)
#     slice_index = info['slice_index']
#     tumor_area = info['tumor_area']
#     print(f'  {dataset}: axial slice {slice_index} - tumor area: {tumor_area}')
#     print(f'    Modalities: {", ".join(modalities)}')

# print(f'\nTotal image files created: {total_images}')
# print(f'All files saved to: {output_folder}')
# print('\nAll PNG files now have BLACK PADDING instead of white!')
# print('\nFiles created (ALL with BLACK padding):')
# for dataset, info in results_black_padding.items():
#     for modality in info['modalities'].keys():
#         filename_base = f'{dataset}_{modality}'
#         print(f'  - {filename_base}_image.png (512x512 pixels, BLACK padding)')
#         print(f'  - {filename_base}_label.png (512x512 pixels, black background)')


In [55]:
# # FINAL: UNIFORM SIZE WITHOUT CROPPING - All images exactly 512x512 pixels
# results_uniform = {}

# print('Creating UNIFORM SIZE (512x512) images - NO CROPPING, preserving full brain...')
# print('=' * 80)

# # Include ALL datasets
# all_datasets = ['ISLES', 'BRATS', 'ISLES2022', 'TBI', 'ATLAS', 'MSSEG', 'TUMOUR2']

# for dataset in all_datasets:
#     print(f'\nProcessing {dataset}...')
    
#     pairs = find_matching_files(dataset, data_folder)
    
#     if len(pairs) == 0:
#         print(f'  No matching pairs found for {dataset}')
#         continue
    
#     best_result = None
#     max_tumor_area = 0
    
#     # Check first 5 files to find best tumor
#     for i, (image_path, label_path) in enumerate(pairs[:5]):
#         print(f'  Checking file {i+1}: {os.path.basename(image_path)}')
        
#         result = find_best_axial_tumor_slice_multimodal(image_path, label_path)
        
#         if len(result) == 5:
#             image_slices, modality_names, label_slice, slice_idx, tumor_area = result
            
#             if image_slices is not None and tumor_area > max_tumor_area:
#                 max_tumor_area = tumor_area
#                 best_result = (image_slices, modality_names, label_slice, slice_idx, tumor_area, image_path, label_path)
#                 print(f'    Found better slice {slice_idx} with tumor area {tumor_area}')
#                 print(f'    Modalities: {modality_names}')
    
#     if best_result is not None:
#         image_slices, modality_names, label_slice, slice_idx, tumor_area, image_path, label_path = best_result
        
#         dataset_results = {
#             'original_image': image_path,
#             'original_label': label_path,
#             'slice_index': slice_idx,
#             'tumor_area': tumor_area,
#             'modalities': {}
#         }
        
#         # Save UNIFORM SIZE (no crop) image for each modality
#         for mod_idx, (image_slice, modality_name) in enumerate(zip(image_slices, modality_names)):
#             print(f'  Creating 512x512 NO-CROP images for {dataset} - {modality_name}')
            
#             img_path, lbl_path = save_uniform_size_images_no_crop(
#                 image_slice, label_slice, dataset, modality_name, slice_idx, output_folder
#             )
            
#             dataset_results['modalities'][modality_name] = {
#                 'image_png': img_path,
#                 'label_png': lbl_path
#             }
        
#         results_uniform[dataset] = dataset_results
        
#         print(f'  ✓ Created 512x512 NO-CROP images for {dataset}')
#         print(f'    Axial slice {slice_idx} with tumor area {tumor_area}')
#         print(f'    Created {len(modality_names)} modality images')
#     else:
#         print(f'  ✗ No suitable tumor found for {dataset}')

# print(f'\n' + '=' * 80)
# print(f'UNIFORM SIZE (512x512) NO-CROP IMAGE CREATION COMPLETE')
# print(f'=' * 80)
# print(f'Successfully created images for {len(results_uniform)} datasets:')

# total_images = 0
# for dataset, info in results_uniform.items():
#     modalities = list(info['modalities'].keys())
#     total_images += len(modalities) * 2  # 2 files per modality (image + label)
#     slice_index = info['slice_index']
#     tumor_area = info['tumor_area']
#     print(f'  {dataset}: axial slice {slice_index} - tumor area: {tumor_area}')
#     print(f'    Modalities: {", ".join(modalities)}')

# print(f'\nTotal image files created: {total_images}')
# print(f'All files saved to: {output_folder}')
# print('\nAll PNG files are now EXACTLY 512x512 pixels with FULL BRAIN preserved!')
# print('\nFiles created (ALL EXACTLY 512x512 pixels, NO CROPPING):')
# for dataset, info in results_uniform.items():
#     for modality in info['modalities'].keys():
#         filename_base = f'{dataset}_{modality}'
#         print(f'  - {filename_base}_image.png (512x512 pixels, full brain)')
#         print(f'  - {filename_base}_label.png (512x512 pixels, black background)')


In [56]:
# 🎯 FINAL PROCESSING: Save each individual modality from each dataset with correct naming
# Uses modality names from config.py Database_config.channels dictionary

# Define datasets to process
all_datasets = ['WMH''BRATS', 'ATLAS', 'MSSEG', 'ISLES', 'TBI', 'ISLES2022', 'TUMOUR2']

# Expected modalities from config.py for verification
config_modalities = {
    'BRATS': ["FLAIR", "T1", "T1c", "T2"],
    'ATLAS': ["T1"],
    'MSSEG': ['FLAIR', "T1", "T1c", "T2", "PD"],
    'ISLES': ["FLAIR", "T1", "T2", "DWI"],
    'TBI': ["FLAIR", "T1", "T2", "SWI"],
    'ISLES2022': ['ADC', 'DWI', 'FLAIR'],  # ⭐ CORRECTED! ADC, DWI, FLAIR (not T1, T2)
    'TUMOUR2': ['T1'],
    'WMH': ['FLAIR','T1']
}

print("Starting image generation for all datasets...")
print(f" Output folder: {output_folder}")

results = []

for dataset in all_datasets:
    print(f'\n Processing {dataset}...')
    expected_mods = config_modalities.get(dataset, [])
    print(f'    Expected modalities from config: {expected_mods}')
    
    pairs = find_matching_files(dataset, data_folder)
    
    if len(pairs) == 0:
        print(f"❌ No matching pairs found for {dataset}")
        continue
    
    print(f"✅ Found {len(pairs)} image-label pairs for {dataset}")
    
    # Find the best image (with largest tumor) from first 5 pairs
    best_tumor_area = 0
    best_result = None
    best_pair = None
    
    for i, (image_path, label_path) in enumerate(pairs[:5]):  # Check first 5 pairs
        print(f"  📋 Checking file {i+1}: {os.path.basename(image_path)}")
        
        # Find the best axial slice with config-based modality naming
        result = find_best_axial_tumor_slice_with_config(image_path, label_path)
        if len(result) == 6 and result[0] is not None:
            image_slices, modality_names, label_slice, slice_idx, tumor_area, patient_name = result
            
            if tumor_area > best_tumor_area:
                best_tumor_area = tumor_area
                best_result = result
                best_pair = (image_path, label_path)
                print(f"    ⭐ New best: slice {slice_idx}, tumor area {tumor_area:.0f}, patient: {patient_name}")
    
    if best_result is not None:
        image_slices, modality_names, label_slice, slice_idx, tumor_area, patient_name = best_result
        image_path, label_path = best_pair
        
        print(f"🎯 Selected best image for {dataset}:")
        print(f"   👤 Patient: {patient_name}")
        print(f"   🧠 Axial slice: {slice_idx}")
        print(f"   🎯 Tumor area: {tumor_area:.0f} pixels")
        print(f"   📊 Detected modalities: {modality_names}")
        
        # ✅ Verify modalities match config
        if modality_names == expected_mods:
            print(f"   ✅ Modalities match config perfectly!")
        else:
            print(f"   ⚠️  Modalities differ from config: expected {expected_mods}, got {modality_names}")
        
        # 💾 Save each individual modality with correct naming
        saved_files = []
        for i, (image_slice, modality_name) in enumerate(zip(image_slices, modality_names)):
            print(f"   💾 Saving {modality_name} modality...")
            
            image_path_saved, label_path_saved = save_uniform_size_images_no_crop(
                image_slice, label_slice, dataset, modality_name, slice_idx, output_folder
            )
            
            if image_path_saved:
                saved_files.append((modality_name, image_path_saved, label_path_saved))
                print(f"      ✅ Saved: {os.path.basename(image_path_saved)} & {os.path.basename(label_path_saved)}")
        
        results.append({
            'dataset': dataset,
            'patient': patient_name,
            'slice_idx': slice_idx,
            'tumor_area': tumor_area,
            'modalities': modality_names,
            'expected_modalities': expected_mods,
            'saved_files': saved_files
        })
        
        print(f"   🎉 Completed {dataset}: {len(saved_files)} modalities saved")
    else:
        print(f"❌ No valid images found for {dataset}")

print(f"\n🎉 PROCESSING COMPLETE!")
print(f"📁 All images saved to: {output_folder}")
print(f"🎯 Successfully processed {len(results)} datasets")

# 📋 FINAL SUMMARY
print("\n" + "="*60)
print("📋 FINAL SUMMARY:")
print("="*60)

for result in results:
    dataset = result['dataset']
    modalities = result['modalities']
    expected = result['expected_modalities']
    
    print(f"\n✅ {dataset}:")
    print(f"   👤 Patient: {result['patient']}")
    print(f"   🎯 Tumor area: {result['tumor_area']:.0f} pixels")
    print(f"   📊 Modalities: {', '.join(modalities)} ({len(modalities)} total)")
    
    # Verify against config
    if modalities == expected:
        print(f"   ✅ Config verification: PERFECT MATCH")
    else:
        print(f"   ⚠️  Config verification: Expected {expected}")
    
    # List saved files
    print(f"   💾 Files saved:")
    for mod_name, img_path, lbl_path in result['saved_files']:
        print(f"      • {mod_name}: {os.path.basename(img_path)} + {os.path.basename(lbl_path)}")

print(f"\n🔍 Verifying uniform dimensions...")
from PIL import Image
sample_files = []
for result in results:
    for modality, img_path, lbl_path in result['saved_files'][:1]:  # Check first file from each dataset
        sample_files.extend([img_path, lbl_path])

if sample_files:
    print("📏 Image dimensions check:")
    all_same_size = True
    reference_size = None
    
    for file_path in sample_files:
        if os.path.exists(file_path):
            img = Image.open(file_path)
            size = img.size
            if reference_size is None:
                reference_size = size
            elif size != reference_size:
                all_same_size = False
            print(f"   {os.path.basename(file_path)}: {size[0]}x{size[1]} pixels")
    
    if all_same_size:
        print(f"   ✅ ALL IMAGES ARE UNIFORM SIZE: {reference_size[0]}x{reference_size[1]} pixels")
    else:
        print(f"   ❌ SIZE MISMATCH DETECTED!")

print("\n🎯 TASK COMPLETED SUCCESSFULLY! 🎯")


Starting image generation for all datasets...
 Output folder: /home/magd6292/Documents/wentian_clone/MultiUnet/dataset_samples

 Processing WMHBRATS...
    Expected modalities from config: []
❌ No matching pairs found for WMHBRATS

 Processing ATLAS...
    Expected modalities from config: ['T1']
✅ Found 654 image-label pairs for ATLAS
  📋 Checking file 1: sub-r001s001_normed.nii.gz
    ⭐ New best: slice 95, tumor area 2325, patient: sub-r001s001_normed
  📋 Checking file 2: sub-r001s002_normed.nii.gz
    ⭐ New best: slice 77, tumor area 2683, patient: sub-r001s002_normed
  📋 Checking file 3: sub-r001s003_normed.nii.gz
  📋 Checking file 4: sub-r001s004_normed.nii.gz
  📋 Checking file 5: sub-r001s005_normed.nii.gz
🎯 Selected best image for ATLAS:
   👤 Patient: sub-r001s002_normed
   🧠 Axial slice: 77
   🎯 Tumor area: 2683 pixels
   📊 Detected modalities: ['Single']
   ⚠️  Modalities differ from config: expected ['T1'], got ['Single']
   💾 Saving Single modality...
      ✅ Saved: ATLAS_Sin